# Spatial preprocessing and travel-time estimation

Curated excerpts of Shangkun Jiang's original research notebook `Code/Travel Time Cal - V2-For use.ipynb`. This notebook preserves the original GIS operations and Type 1/2/3 travel-time calculation for the SW 20th Avenue validation segment. Outputs and personal paths have been removed. Inputs are not distributed.

**Read before running:** this historical analysis snapshot uses a **30 m stop buffer** and a **strictly greater than 15 s dwell rule**. The published description uses a 20 m buffer and “at least 15 s.” These settings have intentionally not been rewritten. Segment/stop row selections, intersection IDs and `pdist` thresholds refer to the original Gainesville feed. They must be audited before use with another feed. The original projected CRS is EPSG:3857; it is preserved here, with its distance distortion noted in the README.

The code requires a compatible GTFS Static feed in `gtfs_static/`, an intersection layer `intersections.geojson`, and historical BusTime Excel records in `realtime/`. The README lists required columns. This is original analysis evidence, not a self-contained publication reproduction. Run the separate `demo/synthetic_demo.py` to inspect the method without research data.


In [ ]:
# Portfolio adapter: configure authorized inputs outside the repository.
from pathlib import Path
import os
if not os.environ.get("GTFS_DATA_ROOT"):
    raise RuntimeError("Set GTFS_DATA_ROOT to a directory containing your authorized input files.")
DATA_ROOT = Path(os.environ["GTFS_DATA_ROOT"]).expanduser()
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 0 (zero-based).
import pandas as pd
import geopandas as gpd
from shapely import geometry
from shapely.geometry import Point, LineString, shape, MultiLineString
import matplotlib.pyplot as plt
import contextily as ctx
import datetime
import matplotlib.dates as mdates
from matplotlib.patches import Circle, FancyArrow
import matplotlib.patches as mpatches
import numpy as np
import math
import statistics
from geopy.distance import geodesic
import os
import json
from matplotlib.dates import date2num, HourLocator, DateFormatter
from datetime import datetime
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy import stats
import random


import warnings
warnings.filterwarnings('ignore')

## GTFS Static and road intersections

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 2 (zero-based).
########################################## Read GTFS Static data ##########################################
static_file_path = str(DATA_ROOT / "gtfs_static")
shapes = '/shapes.txt' # bus routes
stop_times = '/stop_times.txt' # stop sequence
stops = '/stops.txt' # stop location
trips = '/trips.txt' # trip info
routes = '/routes.txt' # draw route

static_shapes = pd.read_csv(static_file_path + shapes)
static_stop_times = pd.read_csv(static_file_path + stop_times)[['trip_id', 'stop_id', 'stop_sequence', 'timepoint']]
static_stops = pd.read_csv(static_file_path + stops)[['stop_id', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon']]
static_trips = (pd.read_csv(static_file_path + trips)).replace(r'\t', '', regex=True)
static_trips = (static_trips[['trip_id', 'route_id', 'trip_headsign', 'direction_id', 'shape_id']]).drop_duplicates(subset='shape_id', keep='first').reset_index(drop=True)
static_routes = pd.read_csv(static_file_path + routes)

# bus stops on each route and their sequency
static_bus_stops = (static_trips.merge(static_stop_times, how='left', on='trip_id')).merge(static_stops, how='left', on='stop_id')

# geo_route shapes: point to line
geo_static_shapes = gpd.GeoDataFrame(data=static_shapes,
                                     geometry=gpd.points_from_xy(static_shapes['shape_pt_lon'], static_shapes['shape_pt_lat']),
                                     crs='EPSG:4326').to_crs('EPSG:3857')
geo_route_shape = geo_static_shapes.groupby(['shape_id'])['geometry'].apply(lambda x: LineString(x.tolist())) ## point -> line
geo_route = (gpd.GeoDataFrame(geo_route_shape)).reset_index()
geo_route = geo_route.set_crs('EPSG:3857')

# geo_bus stops
geo_static_bus_stops = gpd.GeoDataFrame(data=static_bus_stops,
                                        geometry=gpd.points_from_xy(static_bus_stops['stop_lon'], static_bus_stops['stop_lat']),
                                        crs='EPSG:4326').to_crs('EPSG:3857')

# find route color
geo_route['route_short_name'] = -1
for i in range(geo_route.shape[0]):
    geo_route.loc[i, 'route_short_name'] = int(''.join(c for c in geo_route.loc[i,'shape_id'] if c.isdigit()))

geo_route = geo_route.merge(static_routes, on='route_short_name', how='left')[['shape_id', 'geometry', 'route_color']]
# geo_route = geo_route.merge(static_routes, on='route_short_name', how='left')[['shape_id', 'geometry', 'route_color_x']]


################################### Find out intersection on which route ###################################
############ Prerequest: route buffer ############

# route buffer
route_buffer_length = 20

geo_route_buffer = gpd.GeoDataFrame(geo_route.buffer(route_buffer_length))
geo_route_buffer.columns = ['geometry']
geo_route_buffer['shape_id'] = geo_route['shape_id']
geo_route_buffer = geo_route_buffer.set_geometry(geo_route_buffer['geometry']).set_crs('EPSG:3857')

intersection_file_path = str(DATA_ROOT / "intersections.geojson")
geo_intersection = gpd.read_file(intersection_file_path)[['ID', 'Latitude', 'Longitude', 'geometry']].set_crs('EPSG:4326').to_crs('EPSG:3857')

intersection_on_route = gpd.GeoDataFrame()
for route_name in list(geo_route_buffer['shape_id']):
    overlay_result = gpd.overlay(df1 = geo_intersection,
                                 df2 = geo_route_buffer[geo_route_buffer['shape_id'] == route_name],
                                 how = 'intersection')
    intersection_on_route = pd.concat([intersection_on_route, overlay_result], ignore_index=True)

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 5 (zero-based).
"""
投影bus stop和intersection 到bus route上面
"""
################################### Project bus stop and intersection ###################################
dict_projected_stop = {}
dict_projected_intersection = {}

for shape_id in geo_static_bus_stops['shape_id'].unique():
    my_bus_stop = geo_static_bus_stops[geo_static_bus_stops['shape_id'] == shape_id]
    my_bus_route = geo_route[geo_route['shape_id'] == shape_id]
    my_intersection = intersection_on_route[intersection_on_route['shape_id'] == shape_id]
    
    projected_bus_stops = []
    projected_intersections = []
    
    for index, point in my_bus_stop.iterrows():
        point_geometry = point['geometry']
        nearest_point = my_bus_route['geometry'].iloc[0].interpolate(my_bus_route['geometry'].iloc[0].project(point_geometry))
        projected_bus_stops.append(nearest_point)
    geo_projected_bus_stop = gpd.GeoDataFrame(geometry=projected_bus_stops, crs='EPSG:3857')
    dict_projected_stop[shape_id] = geo_projected_bus_stop
    
    for index, point in my_intersection.iterrows():
        point_geometry = point['geometry']
        nearest_point = my_bus_route['geometry'].iloc[0].interpolate(my_bus_route['geometry'].iloc[0].project(point_geometry))
        projected_intersections.append(nearest_point)
    geo_projected_intersection = gpd.GeoDataFrame(geometry=projected_intersections, crs='EPSG:3857')
    dict_projected_intersection[shape_id] = geo_projected_intersection

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 8 (zero-based).
"""
按照segment划分segment
提前要用projected intersection设置一个小的intersection buffer，用于划分
"""

################################### Segment split ###################################
# set a little buffer (r=0.5m): geo_intersection_buffer_usedfor_split_segment
# used for segment split
geo_intersection_buffer_usedfor_split_segment = pd.DataFrame()
geo_stop_buffer = pd.DataFrame()

intersection_buffer_length_usedfor_split_segment = 0.5
stop_buffer_length = 30

for shape_id in dict_projected_intersection.keys():
    this_geo_intersection_buffer_usedfor_split_segment = gpd.GeoDataFrame(dict_projected_intersection[shape_id].buffer(intersection_buffer_length_usedfor_split_segment))
    this_geo_intersection_buffer_usedfor_split_segment.columns = ['geometry']
    this_intersection = intersection_on_route[intersection_on_route['shape_id'] == shape_id]
    this_intersection = this_intersection[['ID', 'Latitude', 'Longitude', 'shape_id']].reset_index(drop=True)
    this_geo_intersection_buffer_usedfor_split_segment = pd.concat([this_geo_intersection_buffer_usedfor_split_segment, this_intersection], axis=1)
    geo_intersection_buffer_usedfor_split_segment = pd.concat([geo_intersection_buffer_usedfor_split_segment, this_geo_intersection_buffer_usedfor_split_segment], ignore_index=True)
    geo_intersection_buffer_usedfor_split_segment = geo_intersection_buffer_usedfor_split_segment.set_geometry(geo_intersection_buffer_usedfor_split_segment['geometry']).set_crs('EPSG:3857')

geo_segment_split_results = pd.DataFrame()
for shape_id in geo_route['shape_id'].unique():
    my_geo_route = geo_route[geo_route['shape_id'] == shape_id]
    my_geo_intersection = geo_intersection_buffer_usedfor_split_segment[geo_intersection_buffer_usedfor_split_segment['shape_id'] == shape_id]
    my_route_segment = gpd.overlay(df1=my_geo_route, df2=my_geo_intersection, how='difference')
    this_my_route_segment = my_route_segment.explode()
    geo_segment_split_results = pd.concat([geo_segment_split_results, this_my_route_segment], ignore_index=True)

################################### Change segment from LINESTRING into buffer ###################################
### Set route buffer ###
route_buffer_length = 20
geo_split_segment_buffer = gpd.GeoDataFrame(geo_segment_split_results.buffer(route_buffer_length))
geo_split_segment_buffer.columns = ['geometry']
geo_split_segment_buffer[['shape_id', 'route_color']] = geo_segment_split_results[['shape_id', 'route_color']]
geo_split_segment_buffer = geo_split_segment_buffer.set_geometry(geo_split_segment_buffer['geometry']).set_crs('EPSG:3857')

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 10 (zero-based).
################################### Bus stop buffer ###################################
for shape_id in dict_projected_stop.keys():
    this_geo_stop_buffer = gpd.GeoDataFrame(dict_projected_stop[shape_id].buffer(stop_buffer_length))
    this_geo_stop_buffer.columns = ['geometry']
    this_geo_stop_buffer[['trip_id', 'route_id', 'trip_headsign', 'direction_id', 'shape_id', 'stop_id', 'stop_sequence', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon']] = geo_static_bus_stops[geo_static_bus_stops['shape_id'] == shape_id][['trip_id', 'route_id', 'trip_headsign', 'direction_id', 'shape_id', 'stop_id', 'stop_sequence', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon']].reset_index(drop=True)
    geo_stop_buffer = pd.concat([geo_stop_buffer, this_geo_stop_buffer], ignore_index=True)
    geo_stop_buffer = geo_stop_buffer.set_geometry(geo_stop_buffer['geometry']).set_crs('EPSG:3857')

## Historical BusTime records

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 14 (zero-based).
# read data
folder_path = str(DATA_ROOT / "realtime")
gtfs_rt_data = pd.DataFrame() # 10.9 - 10.13
file_list = os.listdir(folder_path)
excel_files = [file for file in file_list if file.endswith('.xlsx') or file.endswith('.xls')]

for excel_file in excel_files:
    file_path = os.path.join(folder_path, excel_file)
    df = pd.read_excel(file_path)
    gtfs_rt_data = pd.concat([gtfs_rt_data, df], ignore_index=True) # 1120084

# drop duplicate
gtfs_rt_data = gtfs_rt_data.drop('Unnamed: 0', axis=1)
gtfs_rt_data = gtfs_rt_data.drop_duplicates(keep = 'first', ignore_index=True)

# choose columns
gtfs_rt_data = gtfs_rt_data[['vid', 'tmstmp', 'lat', 'lon', 'pid', 'rt', 'des', 'pdist', 'spd', 'tatripid', 'stsd']]
geo_gtfs_rt_data = gpd.GeoDataFrame(data=gtfs_rt_data,
                                    geometry=gpd.points_from_xy(gtfs_rt_data['lon'], gtfs_rt_data['lat']),
                                    crs='EPSG:4326').to_crs('EPSG:3857')

# change into python.datetime
geo_gtfs_rt_data['tmstmp'] = pd.to_datetime(geo_gtfs_rt_data['tmstmp'])

# sort values & reset index
geo_gtfs_rt_data = geo_gtfs_rt_data.sort_values(by='tmstmp', ascending=True)
geo_gtfs_rt_data = geo_gtfs_rt_data.reset_index(drop=True)

## SW 20th Avenue: complete trips and three travel-time components

The source segment length is 1,673 m. This excerpt retains five original stop selections, trip/date grouping, mph-to-m/s conversion, observed/imputed stop crossing, and interpolation at segment boundaries. Empty/incomplete trips and zero boundary speeds require review with actual inputs; the original loops are retained.

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 31 (zero-based).
# geo_gtfs_rt_data_20I = geo_gtfs_rt_data[geo_gtfs_rt_data['pid'] == 500689].reset_index(drop=True)
# geo_gtfs_rt_data_20I.to_excel('geo_gtfs_rt_data_20I.xlsx', index = False)
geo_gtfs_rt_data_20I = geo_gtfs_rt_data[geo_gtfs_rt_data['rt'] == 20][geo_gtfs_rt_data[geo_gtfs_rt_data['rt'] == 20]['des'] == 'Reitz Union/UF Campus']

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 38 (zero-based).
### bus行程时间的计算，三种不同的时间 ------------- Version 2 --------------

"""for 循环找每个trip 并做如下操作：

1. 找trip：date + tripid；
2. 删除不完整的trip：认为没有完整通过validation route的就是不完整的
3. validation segment和trip内的所有data overlay，找到intersection的data
4. 对intersection的data在与stop buffer overlay，找到intersection的data，并且标注出来属于哪个stop buffer；
5. 分别求解三种travel time

"""

# route length
length = 1673

# 合并路段 和 stop buffer：multi_polygon_Archer 和 geo_multi_polygon_stops_buffer
multi_polygon_Archer = geo_split_segment_buffer[geo_split_segment_buffer['shape_id'] == '20I'].iloc[3:5].unary_union
geo_multi_polygon_Archer = gpd.GeoDataFrame({'geometry': [multi_polygon_Archer]})

# route1I上的所有stop buffer
s1 = geo_stop_buffer[geo_stop_buffer['shape_id'] == '20I'].iloc[8:9]
s2 = geo_stop_buffer[geo_stop_buffer['shape_id'] == '20I'].iloc[9:10]
s3 = geo_stop_buffer[geo_stop_buffer['shape_id'] == '20I'].iloc[10:11]
s4 = geo_stop_buffer[geo_stop_buffer['shape_id'] == '20I'].iloc[11:12]
s5 = geo_stop_buffer[geo_stop_buffer['shape_id'] == '20I'].iloc[12:13]

time_costs = []
t1s = []
t2s = []
t3s = []
timestamps = []
mean_space_speeds = []

for date in geo_gtfs_rt_data_20I['stsd'].unique():
    for trip in geo_gtfs_rt_data_20I[geo_gtfs_rt_data_20I['stsd'] == date]['tatripid'].unique():
        geo_20I_trip_day = geo_gtfs_rt_data_20I[geo_gtfs_rt_data_20I['stsd'] == date][geo_gtfs_rt_data_20I[geo_gtfs_rt_data_20I['stsd'] == date]['tatripid'] == trip]
        geo_20I_trip_day = geo_20I_trip_day.reset_index(drop=True)
        
        if (geo_20I_trip_day.head(1)['pdist'].iloc[0] < 10560) & (geo_20I_trip_day.tail(1)['pdist'].iloc[-1] > 18480):
            # geo_20I_trip_day 与segment buffer sjoin
            point_in_seg = gpd.overlay(df1=geo_20I_trip_day,
                                       df2=geo_multi_polygon_Archer,
                                       how='intersection')
            # geo_20I_trip_day 与stop buffer sjoin
            point_in_stop1 = gpd.overlay(df1=point_in_seg,
                                         df2=s1,
                                         how='intersection')
            point_in_stop1['stop_buffer'] = 1

            point_in_stop2 = gpd.overlay(df1=point_in_seg,
                                         df2=s2,
                                         how='intersection')
            point_in_stop2['stop_buffer'] = 2

            point_in_stop3 = gpd.overlay(df1=point_in_seg,
                                         df2=s3,
                                         how='intersection')
            point_in_stop3['stop_buffer'] = 3

            point_in_stop4 = gpd.overlay(df1=point_in_seg,
                                         df2=s4,
                                         how='intersection')
            point_in_stop4['stop_buffer'] = 4

            point_in_stop5 = gpd.overlay(df1=point_in_seg,
                                         df2=s5,
                                         how='intersection')
            point_in_stop5['stop_buffer'] = 5

#             point_in_stop6 = gpd.overlay(df1=point_in_seg,
#                                          df2=s6,
#                                          how='intersection')
#             point_in_stop6['stop_buffer'] = 6
            
            point_in_all_stop = pd.DataFrame()
            point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop1], ignore_index=True)
            point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop2], ignore_index=True)
            point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop3], ignore_index=True)
            point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop4], ignore_index=True)
            point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop5], ignore_index=True)
#             point_in_all_stop = pd.concat([point_in_all_stop, point_in_stop6], ignore_index=True)
            
            # 所有在segment上的点
            point_in_seg = point_in_seg.merge(point_in_all_stop, how='left', on='tmstmp')
            point_in_seg = point_in_seg.set_geometry(point_in_seg['geometry_x'])
            
    #######-------------------------------------------------------------------#######
            # 求bus stop的第一类时间t1
            t1 = 0
            # 划分出来连续的在stop_buffer之外的数据，作为list放在了seg_all里面
            seg = []
            seg_all = []

            for i in range(point_in_seg.shape[0]):
                if pd.isna(point_in_seg.loc[i, 'stop_buffer']):
                    seg.append(i)
                else:
                    if len(seg) >= 1:
                        seg_all.append(seg)
                    seg = []

            if len(seg) >= 1:
                seg_all.append(seg)

            # travel time第一类时间 t1的计算
            for seg in seg_all:
                t1_travel = (point_in_seg.loc[seg[-1], 'tmstmp'] - point_in_seg.loc[seg[0], 'tmstmp']).seconds
                t1 = t1 + t1_travel
            
    #######-------------------------------------------------------------------#######
            # 求bus stop的第二类时间t2
            # 判断是否bus skipping和由于bus stop所导致的可能延误
            t2 = 0
            point_in_seg['flag'] = point_in_seg['stop_buffer'].isna().astype(int)
            nan_flag_indices = point_in_seg[(point_in_seg['flag'] == 1) & point_in_seg['stop_buffer'].isna()].index
            # 找到前后两个数值之差大于1的索引
            diff_indices = np.where(np.diff(nan_flag_indices) > 1)[0]
            # 初始化结果数组
            result_array = []
            # 提取起始和结束值，然后将它们放到结果数组中
            for index in diff_indices:
                start_value = nan_flag_indices[index]
                end_value = nan_flag_indices[index + 1]
                result_array.append([start_value, end_value])

            for temp in result_array:
                b4_stop_index = temp[0]
                b4_stop = point_in_seg.iloc[[b4_stop_index]].reset_index(drop=True)
                after_stop_index = temp[-1]
                after_stop = point_in_seg.iloc[[after_stop_index]].reset_index(drop=True)

                v = (after_stop['spd_x'].iloc[0] + b4_stop['spd_x'].iloc[0])/2
                # 匀速行驶所用时间
                t_theory = after_stop.distance(b4_stop).iloc[0]/(v*0.44704)
                # 实际行驶时间
                t_real = (after_stop['tmstmp'].iloc[0] - b4_stop['tmstmp'].iloc[0]).seconds
                if t_real - t_theory > 15:
                    t2 = t2 + t_theory
                else:
                    t2 = t2 + t_real
            
    #######-------------------------------------------------------------------#######
            # 求进出intersection所用的时间t3
            # 在segment前的那个点的索引
            index_b4_seg = geo_20I_trip_day [geo_20I_trip_day ['tmstmp']==point_in_seg['tmstmp'].iloc[0]].index-1
            # 在segment后的那个点的索引
            index_after_seg = geo_20I_trip_day [geo_20I_trip_day ['tmstmp']==point_in_seg['tmstmp'][point_in_seg['tmstmp'].shape[0]-1]].index+1

            # 在segment前的那个点
            p_b4_seg = geo_20I_trip_day .iloc[index_b4_seg].reset_index(drop=True)
            # segment中的第一个intersection
            intersection_1st = geo_intersection[geo_intersection['ID'] == 84887632].reset_index(drop=True)

            # 在segment后的那个点
            p_after_seg = geo_20I_trip_day .iloc[index_after_seg].reset_index(drop=True)
            # segment中的最后一个intersection
            intersection_last = geo_intersection[geo_intersection['ID'] == 84895704].reset_index(drop=True)

            # segment中的第一个点
            p1 = point_in_seg.head(1).reset_index(drop=True)
            # segment中的最后一个点
            p_last = point_in_seg.tail(1).reset_index(drop=True)

            ### 进入时间：
            t3_1 = ((p1['tmstmp'].iloc[0]-p_b4_seg['tmstmp'].iloc[0]).seconds) * p1.distance(intersection_1st).iloc[0] / p1.distance(p_b4_seg).iloc[0]
            ### 离开时间：
            t3_2 = ((p_after_seg['tmstmp'].iloc[0]-p_last['tmstmp'].iloc[0]).seconds) * p_last.distance(intersection_last).iloc[0] / p_last.distance(p_after_seg).iloc[0]
            # 进出intersection的时间：
            t3 = t3_1 + t3_2
            
            
            t_all = t1 + t2 + t3
            timestamp = point_in_seg.loc[0, 'tmstmp']
            mean_space_speed = length / t_all
            time_costs.append(t_all)
            timestamps.append(timestamp)
            mean_space_speeds.append(mean_space_speed)
            t1s.append(t1)
            t2s.append(t2)
            t3s.append(t3)
        else:
            continue

df = pd.DataFrame({'timestamps': timestamps, 't1': t1s, 't2': t2s, 't3': t3s, 'time_costs': time_costs, 'mean_space_speeds': mean_space_speeds})

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 39 (zero-based).
df.to_excel(str(OUTPUT_DIR / "sw20_travel_times.xlsx"), index = False)